# Drone Audio Classification â€” Kaggle Training

**Before running:**
- Settings > Accelerator > GPU T4 x2
- Settings > Internet > On
- Add dataset: `aranagnost/drone-audio-dataset`

## 1. Verify GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2. Clone the repo

In [ ]:
import os
import sys

REPO_DIR = '/kaggle/working/drone-audio-classification'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aranagnost/drone-audio-classification.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 3. Check dataset path

In [7]:
import os
print(os.listdir('/kaggle/input/datasets/aranagnost/drone-audio-dataset'))

['datasets']


In [9]:
DATASET_ROOT = '/kaggle/input/datasets/aranagnost/drone-audio-dataset/datasets/Drone_Audio_Dataset/audio'
print('Dataset root exists:', os.path.exists(DATASET_ROOT))
print('Subdirs:', os.listdir(DATASET_ROOT))

Dataset root exists: True
Subdirs: ['4_motors', '8_motors', '2_motors', 'not_a_drone', '6_motors']


## 4. Create artifacts directory

In [ ]:
os.makedirs('artifacts/checkpoints', exist_ok=True)
CACHE_DIR = '/kaggle/working/ast_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

## 5. Install transformers (for AST)

In [ ]:
!pip install -q transformers

## 6. Train AST Stage 2 (motor count: 2/4/6/8 motors)

**Changes vs v5:**
- **MLP head** (`--mlp_head`): replaces the linear classifier (768→4) with a 2-layer MLP (768→256→GELU→dropout→4). v5 plateaued with 6_motors F1=0.25 because the 6/8 boundary is non-linear in the frozen feature space — a linear head can't draw that boundary. The MLP adds the capacity to find it.
- Checkpoint saved as `stage2_ast_v6.pt`

**Carried over from v5:**
- Partial unfreeze: last 3 transformer blocks + layernorm (`--unfreeze_top_n 3`)
- LR 5e-6, dropout 0.3
- All quality segments (q1–q5)
- Stratified splits + SpecAugment

In [ ]:
os.environ['PYTHONPATH'] = REPO_DIR
!python training/train_ast.py --task stage2 --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --batch 16 --num_workers 4 --out artifacts/checkpoints/stage2_ast_v6.pt --freeze_encoder --unfreeze_top_n 3 --lr 5e-6 --dropout 0.3 --mlp_head

## 7. Evaluate AST (Stage 2)

In [ ]:
!python training/train_ast.py --task stage2 --eval --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --out artifacts/checkpoints/stage2_ast_v6.pt --mlp_head

## 8. Save checkpoint and send via Telegram

In [ ]:
import os, requests
from kaggle_secrets import UserSecretsClient

secrets    = UserSecretsClient()
TG_TOKEN   = secrets.get_secret("TG_TOKEN")
TG_CHAT_ID = secrets.get_secret("TG_CHAT_ID")
CKPT_NAME  = "stage2_ast_v6.pt"
src        = f"/kaggle/working/{CKPT_NAME}"

if not os.path.exists(src):
    print(f"[WARN] {src} not found — run the save cell first.")
else:
    with open(src, "rb") as f:
        r = requests.post(
            f"https://api.telegram.org/bot{TG_TOKEN}/sendDocument",
            data={"chat_id": TG_CHAT_ID, "caption": f"Training done — {CKPT_NAME}"},
            files={"document": f},
        )
    print("Sent to Telegram!" if r.ok else f"Failed: {r.text}")